In [1]:
# 딥러닝
import pandas as pd
import torch
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from ydata_profiling import ProfileReport


In [2]:
# 1. 전처리된 데이터 불러오기
# 머신러닝에서 만들어진 데이터 불러와
train_df = pd.read_pickle("../pkl_file/machine_data/train_machineData.pkl")
test_df = pd.read_pickle("../pkl_file/machine_data/test_machineData.pkl")

print(f"✅ 데이터 로드 완료: 학습용 {len(train_df)}건 / 테스트용 {len(test_df)}건")
print("-" * 30)

# 각 데이터 합계
train_total = len(train_df)
test_total = len(test_df)

# 라벨링 잘됐는지 확인
compare_df = pd.DataFrame({
    f'Train(학습용) {train_total}': train_df['case_type'].value_counts(),
    f'Test(테스트용) {test_total}': test_df['case_type'].value_counts()
})
print(compare_df)

✅ 데이터 로드 완료: 학습용 48308건 / 테스트용 5435건
------------------------------
           Train(학습용) 48308  Test(테스트용) 5435
case_type                                   
형사소송                  37831             4542
행정소송                   9433              801
민사/가사소송                1044               92


In [3]:
# 2. 라벨 숫자 변환
train_df['case_type'] = train_df['case_type'].astype('category')
test_df['case_type'] = test_df['case_type'].astype('category')

num_labels = len(train_df['case_type'].cat.categories)
train_labels = train_df['case_type'].cat.codes.tolist()
test_labels = test_df['case_type'].cat.codes.tolist()


# 3. 클래스 가중치 계산 (데이터 불균형 해결용)
class_weights = compute_class_weight(
    class_weight='balanced', classes=np.unique(train_labels),
    y=train_labels
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print(f"✅ 클래스 개수: {num_labels}")
print(f"✅ 계산된 가중치: {class_weights}")
# 민사/가사가 낮아서 그쪽에 가중치를 줌.
# 카테고리 순서(0번부터 순서대로) 출력
print(train_df['case_type'].cat.categories.tolist())

✅ 클래스 개수: 3
✅ 계산된 가중치: tensor([15.4240,  1.7071,  0.4256])
['민사/가사소송', '행정소송', '형사소송']


In [4]:
# 4. 토큰화 (학습시키기 위해 서브워드 단위로 쪼갬. 멈춤 방지를 위해 max_length를 512 설정)
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
train_encodings = tokenizer(train_df['text'].tolist(), truncation=True, 
                            padding='max_length', max_length=512)
test_encodings = tokenizer(test_df['text'].tolist(), truncation=True,
                           padding='max_length', max_length=512)
print("✅ 토크나이징 완료!")

# [토크나이징 샘플 확인]
sample_index = 0  # 첫 번째 데이터 확인

# 1. 숫자로 변환된 결과 (input_ids)
sample_ids = train_encodings['input_ids'][sample_index]

# 2. 숫자를 다시 단어 조각(Tokens)으로 복원
sample_tokens = tokenizer.convert_ids_to_tokens(sample_ids)

print(f"\n --- [샘플 데이터 토크나이징 결과] ---")
print(f"원문 텍스트 일부: {train_df['text'].iloc[sample_index][:100]}...")
print("-" * 50)
print(f"1. 복원된 토큰(조각): \n{sample_tokens[:30]} ...") 
print("-" * 50)
print(f"2. 인코딩된 숫자(ID): \n{sample_ids[:30]} ...")

✅ 토크나이징 완료!

 --- [샘플 데이터 토크나이징 결과] ---
원문 텍스트 일부: 기소유예처분취소 피청구인이 청구인의 신호위반 책임을 인정한 구체적인 증거들은 무엇이었나요? 피청구인은 청구인에게 신호위반의 책임을 인정하기 위해 스타렉스 승합차의 운전자와 탑승자의...
--------------------------------------------------
1. 복원된 토큰(조각): 
['[CLS]', '기소', '##유예', '##처분', '##취', '##소', '피', '##청', '##구', '##인', '##이', '청구인', '##의', '신호', '##위', '##반', '책임', '##을', '인정', '##한', '구체', '##적인', '증거', '##들', '##은', '무엇', '##이', '##었', '##나', '##요'] ...
--------------------------------------------------
2. 인코딩된 숫자(ID): 
[2, 5927, 13145, 12892, 2586, 2024, 1882, 2270, 2251, 2179, 2052, 21718, 2079, 6320, 2090, 2536, 3998, 2069, 4089, 2470, 4468, 31221, 5331, 2031, 2073, 3890, 2052, 2359, 2075, 2182] ...


In [5]:
# 5. Dataset 및 WeightedTrainer 정의 (사용자 기존 코드 유지)
class LegalDataset(Dataset):
    def __init__(self, encodings, labels, win_rates, sentences, fines, risks):
        self.encodings = encodings
        self.labels = labels
        self.win_rates = win_rates
        self.sentences = sentences
        self.fines = fines
        self.risks = risks
        
    def __getitem__(self, idx):
        # 학습 시 하나씩 꺼내서 텐서(Tensor/숫자를 담는 다차원 상자) 형태로 변환해주는 역할
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # labels는 내부 로직상 필요하므로 유지하되, 나중에 손실 계산에서 비중을 조절할 수 있습니다.
        item['labels'] = torch.tensor(self.labels[idx])
        item['win_rate'] = torch.tensor(self.win_rates[idx], dtype=torch.float)
        item['sentence'] = torch.tensor(self.sentences[idx], dtype=torch.float)
        item['fine'] = torch.tensor(self.fines[idx], dtype=torch.float)
        item['risk'] = torch.tensor(self.risks[idx], dtype=torch.float)
        return item
    
    def __len__(self):
        return len(self.labels)

# 데이터셋 객체 생성(데이터 가져와)
# 데이터셋 객체 생성 시 머신러닝에서 만든 컬럼넣기
train_dataset = LegalDataset(
    train_encodings, train_labels,
    train_df['win_rate'].values, 
    train_df['sentence_years'].values,
    train_df['fine_amount'].values, 
    train_df['risk_score'].values
)
test_dataset = LegalDataset(
    test_encodings, test_labels,
    test_df['win_rate'].values, 
    test_df['sentence_years'].values,
    test_df['fine_amount'].values, 
    test_df['risk_score'].values
)


# 텐서 형태 변환한거 확인하기!
# 학습 데이터셋에서 첫 번째 샘플(인덱스 0) 하나를 꺼내봅니다.
sample_item = train_dataset[0]

print("--- 텐서 변환 결과 확인 ---")
for key, value in sample_item.items():
    print(f"🔹 필드명: {key}")
    print(f"   - 데이터 타입: {type(value)}") # torch.Tensor인지 확인
    print(f"   - 모양(Shape): {value.shape}")
    
    # 데이터가 0차원(스칼라-> 한개의 숫자)인지 확인하여 출력 방식 변경
    if value.dim() == 0:
        print(f"   - 실제 값: {value.item()}") # 숫자 하나만 출력
    else:
        print(f"   - 실제 값(일부): {value[:10].tolist()}...")# 슬라이싱 후 리스트로 변환
    print("-" * 30)

# 디코딩 확인
decoded_text = tokenizer.decode(sample_item['input_ids'], skip_special_tokens=True)
print(f"✅ 실제 모델이 읽는 텍스트(해석): {decoded_text[:100]}...")

--- 텐서 변환 결과 확인 ---
🔹 필드명: input_ids
   - 데이터 타입: <class 'torch.Tensor'>
   - 모양(Shape): torch.Size([512])
   - 실제 값(일부): [2, 5927, 13145, 12892, 2586, 2024, 1882, 2270, 2251, 2179]...
------------------------------
🔹 필드명: token_type_ids
   - 데이터 타입: <class 'torch.Tensor'>
   - 모양(Shape): torch.Size([512])
   - 실제 값(일부): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]...
------------------------------
🔹 필드명: attention_mask
   - 데이터 타입: <class 'torch.Tensor'>
   - 모양(Shape): torch.Size([512])
   - 실제 값(일부): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...
------------------------------
🔹 필드명: labels
   - 데이터 타입: <class 'torch.Tensor'>
   - 모양(Shape): torch.Size([])
   - 실제 값: 2
------------------------------
🔹 필드명: win_rate
   - 데이터 타입: <class 'torch.Tensor'>
   - 모양(Shape): torch.Size([])
   - 실제 값: 50.0
------------------------------
🔹 필드명: sentence
   - 데이터 타입: <class 'torch.Tensor'>
   - 모양(Shape): torch.Size([])
   - 실제 값: 0.0
------------------------------
🔹 필드명: fine
   - 데이터 타입: <class 'torch.Tensor'>
   - 모양(

In [6]:
# 가중치를 적용한 커스텀 트레이너 클래스(가중치 계산 설계도)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # labels = inputs.get("labels")
        outputs = model(**inputs)
        # logits = outputs.get("logits")
        loss = outputs.get("loss")
        
        # 데이터 불균형을 해결하기 위해 미리 만든 class_weights 적용
        # loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        # loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

print("설계도 완성 및 실제 데이터셋(train/test) 생성 완료!")

설계도 완성 및 실제 데이터셋(train/test) 생성 완료!


In [7]:
# 가중치 설계도 확인

# 1. 라벨 이름 목록 준비
if 'case_type' in train_df.columns and hasattr(train_df['case_type'], 'cat'):
    class_names = train_df['case_type'].cat.categories.tolist()
    # 리스트로 변환하여 에러 방지
    
else:
    class_names = [f"Class_{i}" for i in range(num_labels)]

print("--- ⚖️ WeightedTrainer가 사용할 가중치 설계도 ---")

--- ⚖️ WeightedTrainer가 사용할 가중치 설계도 ---


In [8]:
# 가중치값 출력
weights_list = class_weights.tolist()
for i, weight in enumerate(weights_list):
    name = class_names[i]
    print(f"📍 라벨 {i} [{name:15s}] 가중치: {weight:.4f}")

# 2. 가중치 해석 가이드
# .argmax() 뒤에 .item()을 붙여서 '순수한 인덱스 숫자'만 추출.
max_idx = class_weights.argmax().item()
min_idx = class_weights.argmin().item()

print("\n💡 가중치 해석 가이드:")
print(f"✅ 가중치가 가장 높은 것: {class_names[max_idx]} (데이터가 가장 적음 -> 집중 학습)")
print(f"✅ 가중치가 가장 낮은 것: {class_names[min_idx]} (데이터가 가장 많음 -> 일반 학습)")

📍 라벨 0 [민사/가사소송        ] 가중치: 15.4240
📍 라벨 1 [행정소송           ] 가중치: 1.7071
📍 라벨 2 [형사소송           ] 가중치: 0.4256

💡 가중치 해석 가이드:
✅ 가중치가 가장 높은 것: 민사/가사소송 (데이터가 가장 적음 -> 집중 학습)
✅ 가중치가 가장 낮은 것: 형사소송 (데이터가 가장 많음 -> 일반 학습)


In [9]:
# 채점 함수 정의 (Metrics)
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    
    # predictions가 튜플인 경우 logits만 추출
    if isinstance(pred.predictions, tuple): #데이터 타입이 튜플인지 확인해봐
        logits = pred.predictions[-1]  # 마지막 요소가 logits
    else:
        logits = pred.predictions
        
    # preds = pred.predictions.argmax(-1)
    preds = logits.argmax(-1)
     
    # 정확도(Accuracy)와 불균형을 고려한 F1-score 계산
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro') 
    
    return {
        'accuracy': acc,
        'f1': f1,
    }

In [10]:
# 모델 불러오기
# model = AutoModelForSequenceClassification.from_pretrained("klue/bert-base", num_labels=num_labels).to(device)

import torch.nn as nn
from transformers import BertModel

class MultiTaskLegalBERT(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.config = self.bert.config
        hidden_size = self.bert.config.hidden_size
        
        # 수치 예측용 4가지 헤드 (Regression)
        self.win_rate_head = nn.Linear(hidden_size, 1)
        self.sentence_head = nn.Linear(hidden_size, 1)
        self.fine_head = nn.Linear(hidden_size, 1)
        self.risk_head = nn.Linear(hidden_size, 1)
        
        # 소송 분류는 내부 학습용으로만 유지 (BERT가 법률 맥락을 파악하는 데 도움을 줌)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.num_labels = num_labels

    @property
    def device(self):
        return next(self.parameters()).device

    
    def forward(self, input_ids, attention_mask, labels=None, win_rate=None, sentence=None, fine=None, risk=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output # CLS 토큰의 결과물
        
        # 각 수치 예측
        pred_win = self.win_rate_head(pooled_output).squeeze(-1)
        pred_sent = self.sentence_head(pooled_output).squeeze(-1)
        pred_fine = self.fine_head(pooled_output).squeeze(-1)
        pred_risk = self.risk_head(pooled_output).squeeze(-1)
        logits = self.classifier(pooled_output)
        
        loss = None
        if win_rate is not None:
            mse_fct = nn.MSELoss()
            # 4가지 수치에 대한 오차 합산
            loss = mse_fct(pred_win, win_rate) + \
                   mse_fct(pred_sent, sentence) + \
                   mse_fct(pred_fine, fine) + \
                   mse_fct(pred_risk, risk)
            
            # 소송 유형 분류 손실도 조금 추가해서 학습을 도움 (가중치 0.1)
            loss_fct = nn.CrossEntropyLoss()
            loss += 0.1 * loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            
        return {"loss": loss, "win_rate": pred_win, "sentence": pred_sent, "fine": pred_fine, "risk": pred_risk, "logits": logits}

# 모델 초기화
model = MultiTaskLegalBERT("klue/bert-base", num_labels=num_labels).to(device)



# 메모리를 많이 차지해서 커널충돌이 발생함. -> 아래 두개 코드로 
# 메모리(RAM)부족 방지/필요할때 다시 계산해서 메모리 사용량 줄임
# model 내부의 bert 모델에 있는 메서드를 호출해야 합니다.
model.bert.gradient_checkpointing_enable()
model.to(device)

print("모델 불러오기 완료!")

모델 불러오기 완료!


In [11]:
import random
from torch.utils.data import Subset
import psutil
import time

In [12]:
# 데이터셋 생성 함수
def create_sample_dataset(dataset, sample_ratio=0.1):
    """데이터의 일부만 추출"""
    total_size = len(dataset)
    sample_size = int(total_size * sample_ratio)
    indices = random.sample(range(total_size), sample_size)
    return Subset(dataset, indices)

In [13]:
# 메모리 확인 함수
def check_memory_before_training():
    mem = psutil.virtual_memory()
    available_gb = mem.available / (1024**3)
    
    print(f"💾 현재 사용 가능한 메모리: {available_gb:.1f} GB")
    
    if available_gb < 2.0:
        print("⚠️ 경고: 메모리 부족! 다른 프로그램을 종료하세요.")
        return False
    elif available_gb < 4.0:
        print("⚠️ 주의: 메모리 여유가 적습니다.")
        return True
    else:
        print("✅ 메모리 충분합니다.")
        return True

In [14]:
# 테스트
MODE_1_args = TrainingArguments(
    output_dir="./mode1_quick_test",
    eval_strategy="steps",
    eval_steps=5,
    
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    
    num_train_epochs=1,
    max_steps=10,  # 10 스텝만!
    
    learning_rate=2e-5,
    logging_steps=2,
    save_steps=10,
    
    fp16=False,  # CPU는 False
    dataloader_num_workers=0,
    report_to="none",
)

In [15]:
# 검증
MODE_2_args = TrainingArguments(
    output_dir="./mode2_quick_validation",
    eval_strategy="steps",
    eval_steps=25,
    
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    
    num_train_epochs=1,
    max_steps=100,
    
    learning_rate=2e-5,
    warmup_steps=10,
    logging_steps=10,
    save_steps=50,
    save_total_limit=1,
    
    fp16=False,
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
)

In [16]:
# 실용 모델
MODE_3_args = TrainingArguments(
    output_dir="./mode3_practical_model",
    eval_strategy="steps",
    eval_steps=100,
    
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    
    num_train_epochs=1,
    
    learning_rate=2e-5,
    warmup_steps=50,
    weight_decay=0.01,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    
    fp16=False,
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

In [17]:
print("테스트 시작")

# 모드1 데이터 100% 사용
# 메모리 체크
if not check_memory_before_training():
    raise RuntimeError("❌ 메모리 부족!")

# 학습실행함수
print(f"\n📊 학습 설정")
print(f"  - 학습 데이터: {len(train_dataset):,}개")
print(f"  - 평가 데이터: {len(test_dataset):,}개")

# 트레이너 생성
trainer = WeightedTrainer(
    model=model,
    args=MODE_1_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)
print(f"\n 학습 시작!\n")

# 학습실행
import time
start_time = time.time()
trainer.train()

# 소요 시간 계산
elapsed = time.time() - start_time
hours = int(elapsed // 3600)
minutes = int((elapsed % 3600) // 60)

print(f"\n✅ 학습 완료!")
print(f"⏱️  소요 시간: {hours}시간 {minutes}분")

테스트 시작
💾 현재 사용 가능한 메모리: 10.8 GB
✅ 메모리 충분합니다.

📊 학습 설정
  - 학습 데이터: 48,308개
  - 평가 데이터: 5,435개

 학습 시작!



c:\Users\ASUS\pj_hj\Frondend\Ai\.venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy,F1
5,4562498879488.000000,115086031060992.000000,0.148114,0.092141


c:\Users\ASUS\pj_hj\Frondend\Ai\.venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
# 평가
print(f"\n📊 최종 평가 중...")
eval_results = trainer.evaluate()

print(f"\n🎯 [평가 결과]")
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

# 모델 저장
# save_path = f"./saved_{mode.lower()}"
save_path = "./saved_mode1"
print(f"\n💾 모델 저장 중: {save_path}")

# 추가
# 커스텀 모델이므로 torch.save 사용
os.makedirs(save_path, exist_ok=True)

# PyTorch 방식으로 모델 저장
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'num_labels': num_labels,
        'class_names': train_df['case_type'].cat.categories.tolist(),
        'class_weights': class_weights.cpu().numpy()
    }
}, os.path.join(save_path, 'pytorch_model.bin'))

# model.save_pretrained(save_path)     
tokenizer.save_pretrained(save_path)
print(f"✅ 저장 완료!")


📊 최종 평가 중...


c:\Users\human-24\git\Ai\.venv\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



🎯 [평가 결과]
  eval_loss: 115086031060992.0000
  eval_accuracy: 0.1062
  eval_f1: 0.1119
  eval_runtime: 2820.6488
  eval_samples_per_second: 1.9270
  eval_steps_per_second: 0.2410
  epoch: 0.0017

💾 모델 저장 중: ./saved_mode1
✅ 저장 완료!


In [ ]:
# 결과 저장
import pickle

results_to_save = {
    'eval_results': eval_results,
    'model_config': {
        'num_labels': num_labels,
        'class_names': train_df['case_type'].cat.categories.tolist(),
        'class_weights': class_weights.cpu().numpy()
    },
    'training_history': trainer.state.log_history
}

with open('../pkl_file/deep_data/train_deep_data.pkl', 'wb') as f:
    pickle.dump(results_to_save, f)

print("✅ 학습 결과가 train_deep_data.pkl에 저장되었습니다!")

✅ 학습 결과가 train_deep_data.pkl에 저장되었습니다!


In [ ]:
# 20번 셀 - MODE_2 실행
print("="*70)
print("🏃 MODE 2 실행")
print("="*70)

# 메모리 체크
if not check_memory_before_training():
    raise RuntimeError("❌ 메모리 부족!")

# MODE_2 설정
train_data_2 = create_sample_dataset(train_dataset, 0.05) #데이터 5%만 사용

print(f"\n📊 학습 설정")
print(f"  - 학습 데이터: {len(train_data_2):,}개 (5% 샘플)")
print(f"샘플링: {0.05*100}%")
print(f"  - 평가 데이터: {len(test_dataset):,}개")

🏃 MODE 2 실행
💾 현재 사용 가능한 메모리: 4.8 GB
✅ 메모리 충분합니다.

📊 학습 설정
  - 학습 데이터: 2,415개 (5% 샘플)
샘플링: 5.0%
  - 평가 데이터: 5,435개


In [ ]:
# 트레이너 생성
trainer_2 = WeightedTrainer(
    model=model,
    args=MODE_2_args,
    train_dataset=train_data_2,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print(f"\n MODE_2 학습 시작!\n")

# 학습 실행
start_time_2 = time.time()
trainer_2.train()

# 소요 시간
elapsed_2 = time.time() - start_time_2
hours = int(elapsed_2 // 3600)
minutes = int((elapsed_2 % 3600) // 60)
print(f"\n✅ 학습 완료! ⏱️ {hours}시간 {minutes}분")


 MODE_2 학습 시작!



c:\Users\human-24\git\Ai\.venv\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,F1
25,845311010406.400024,115086031060992.000000,0.024287,0.040199


In [ ]:
# 평가 및 저장
eval_results_2 = trainer_2.evaluate()
print(f"\n🎯 MODE_2 평가 결과:")
for key, value in eval_results_2.items():
    print(f"  {key}: {value:.4f}")

# 모델 저장
save_path_2 = "./saved_mode2"
os.makedirs(save_path_2, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'num_labels': num_labels,
        'class_names': train_df['case_type'].cat.categories.tolist(),
        'class_weights': class_weights.cpu().numpy()
    }
}, os.path.join(save_path_2, 'pytorch_model.bin'))

tokenizer.save_pretrained(save_path_2)
print("✅ MODE_2 완료!")

In [ ]:
# 결과 저장
results_2 = {
    'eval_results': eval_results_2,
    'model_config': {
        'num_labels': num_labels,
        'class_names': train_df['case_type'].cat.categories.tolist(),
        'class_weights': class_weights.cpu().numpy()
    },
    'training_history': trainer_2.state.log_history,
    'elapsed_time': elapsed_2
}

with open('../pkl_file/deep_data/train_deep_data_model2.pkl', 'wb') as f:
    pickle.dump(results_2, f)

print("✅ MODE_2 완료 및 저장!")

In [ ]:
# 새 셀 - MODE_3 실행
print("\n MODE 3: 실용 모델 학습")
print("="*70)

# 메모리 체크
if not check_memory_before_training():
    raise RuntimeError("❌ 메모리 부족!")

# MODE_3 설정 (10% 샘플)
train_data_3 = create_sample_dataset(train_dataset, 0.10)
sample_ratio_3 = 0.10

print(f"\n📊 학습 설정")
print(f"  - 학습 데이터: {len(train_data_3):,}개")
print(f"샘플링: {sample_ratio_3*100}%")  # 10.0% 출력
print(f"  - 평가 데이터: {len(test_dataset):,}개")

In [ ]:
# 트레이너 생성
trainer_3 = WeightedTrainer(
    model=model,
    args=MODE_3_args,  # 더 많은 스텝, warmup 등
    train_dataset=train_data_3,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print(f"\n MODE_3 학습 시작!\n")

# 학습 실행
start_time_3 = time.time()
trainer_3.train()

# 소요 시간
elapsed_3 = time.time() - start_time_3
hours = int(elapsed_3 // 3600)
minutes = int((elapsed_3 % 3600) // 60)
print(f"\n✅ 학습 완료! ⏱️ {hours}시간 {minutes}분")

In [ ]:
# 평가
eval_results_3 = trainer_3.evaluate()
print(f"\n🎯 MODE_3 평가 결과:")
for key, value in eval_results_3.items():
    print(f"  {key}: {value:.4f}")

# 모델 저장
save_path_3 = "./saved_mode3"
os.makedirs(save_path_3, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'num_labels': num_labels,
        'class_names': train_df['case_type'].cat.categories.tolist(),
        'class_weights': class_weights.cpu().numpy()
    }
}, os.path.join(save_path_3, 'pytorch_model.bin'))

tokenizer.save_pretrained(save_path_3)
print("모델 저장완료!")

In [ ]:
# 결과 저장
import pickle
results_3 = {
    'eval_results': eval_results_3,
    'model_config': {
        'num_labels': num_labels,
        'class_names': train_df['case_type'].cat.categories.tolist(),
        'class_weights': class_weights.cpu().numpy()
    },
    'training_history': trainer_3.state.log_history,
    'elapsed_time': elapsed_3
}

with open('../pkl_file/deep_data/train_deep_data_model3.pkl', 'wb') as f:
    pickle.dump(results_3, f)

print("✅ MODE_3 완료 및 저장!")

In [ ]:
# print("🏃 MODE 2: 빠른 검증 (약 30분)")
# args = MODE_2_args
# train_data = create_sample_dataset(train_dataset, 0.05)
# sample_ratio = 0.05

# elif mode == "MODE_3":
# print("🎓 MODE 3: 실용 모델 (약 3시간)")
# args = MODE_3_args
# train_data = create_sample_dataset(train_dataset, 0.10)
# sample_ratio = 0.10
# else:
# raise ValueError("mode는 'MODE_1', 'MODE_2', 'MODE_3' 중 하나")

# print("="*70)


# return trainer, eval_results

# except Exception as e:
# print(f"\n❌ 에러: {e}")
# raise 

# trainer, eval_results = run_training('MODE_1')  # 또는 MODE_2, MODE_3

In [ ]:
# # 학습 옵션 설정
# training_args = TrainingArguments(
#     output_dir="./results",            # 결과 저장 폴더
#     eval_strategy="steps",             # 평가 기준 (steps)
#     per_device_train_batch_size=4,    # 한 번에 4개씩
#     gradient_accumulation_steps=4,    # 4개 실행해
#     fp16=True,
#     max_grad_norm=1.0,                 # 안정성 확보
    
#     num_train_epochs=3,                # 전체 데이터 반복 횟수
#     learning_rate=2e-5,                # 학습률
    
#     logging_steps=100,                 # 100단계마다 로그 출력(잘되는지 확인용도)
#     save_steps=1000,                    # 1000단계마다 모델 저장
#     eval_steps=1000,                     # 평가 빈도
#     save_total_limit=2,                  # 최신 2개 체크포인트만 유지
    
#     # 성능 최적화
#     dataloader_num_workers=2,            # 데이터 로딩 병렬화
#     load_best_model_at_end=True,       # 가장 성능 좋은 모델 최종 선택
#     metric_for_best_model="f1",
#     greater_is_better=True,

#     report_to="none",
#     disable_tqdm=False,                  # 진행률 표시
# )
# print("학습 환경 설정 완료")

In [ ]:
# # 빠른 테스트용 설정
# # 전체 데이터로 학습 전 작은 샘플로 먼저 테스트
# quick_test_args = TrainingArguments(
#     output_dir="./quick_test",
#     eval_strategy="steps",
#     per_device_train_batch_size=8,
#     gradient_accumulation_steps=2,
#     fp16=True,
#     num_train_epochs=1,                  # 1 에폭만
#     max_steps=100,                       # 100 스텝만 실행
#     learning_rate=2e-5,
#     logging_steps=10,
#     eval_steps=50,
#     save_steps=50,
#     report_to="none",
# )

In [ ]:
# # 데이터 샘플링 (빠른 테스트용)
# # 전체 학습 전 10%만 사용해서 테스트
# def create_sample_dataset(dataset, sample_ratio=0.1):
#     """데이터셋의 일부만 추출"""
#     import random
#     total_size = len(dataset)
#     sample_size = int(total_size * sample_ratio)
#     indices = random.sample(range(total_size), sample_size)
    
#     # 서브셋 생성
#     from torch.utils.data import Subset
#     return Subset(dataset, indices)


In [ ]:
# # 트레이너 생성 및 학습가동
# trainer = WeightedTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=test_dataset,
#     compute_metrics=compute_metrics #채점기
# )

# print("🚀 학습 시작")
# print(f"📊 총 학습 데이터: {len(train_dataset):,}건")
# print(f"📊 총 평가 데이터: {len(test_dataset):,}건")
# print(f"⏱️  예상 시간: 배치크기와 하드웨어에 따라 3-10시간")

# # trainer.train()
# # print(" 학습완료")

In [ ]:
# # 학습결과 확인 및 샘플 테스트

# import torch.nn.functional as F

# def predict_legal_case(text):
#     # 모델을 평가 모드로 전환
#     model.eval()
    
#     # 입력 텍스트 토큰화
#     inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    
#     # 예측
#     with torch.no_grad():
#         outputs = model(**inputs)
#         logits = outputs.logits
#         # 확률값으로 변환
#         probs = F.softmax(logits, dim=-1)
#         # 가장 높은 확률의 인덱스 추출
#         pred_idx = torch.argmax(probs, dim=-1).item()
    
#     # 결과 출력
#     class_names = train_df['case_type'].cat.categories.tolist()
#     confidence = probs[0][pred_idx].item() * 100
    
#     print(f"\n[입력 문장]: {text[:50]}...")
#     print(f"[예측 결과]: {class_names[pred_idx]} ({confidence:.2f}% 확신)")


In [ ]:
# 윗셀 대신 이걸로 변경
import torch.nn.functional as F

def predict_full_analysis(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)#.to(device)
    
    # token_type_ids 제거
    inputs = {k: v.to(device) for k, v in inputs.items() if k != 'token_type_ids'}
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    print("\n" + "="*50)
    print("🏛️ [BERT 법률 문맥 분석 결과]")
    print("="*50)
    print(f"📊 예상 승소율: {max(0, min(100, outputs['win_rate'].item())):.1f}%")
    
    # 형량이 0.1년 이상일 때만 출력
    if outputs['sentence'].item() > 0.1:
        print(f"⚖️ 예상 형량: {outputs['sentence'].item():.1f}년")
    
    # 벌금이 1만원 이상일 때만 출력
    if outputs['fine'].item() > 10000:
        print(f"💰 예상 벌금: {outputs['fine'].item():,.0f}원")
    
    print(f"⚠️ 위험도 점수: {max(0, min(100, outputs['risk'].item())):.1f}/100")
    print("="*50)

# 실행 예시
test_text = "저는 회사에서 부당해고를 당했습니다. 퇴직금 500만원도 못 받았습니다."
predict_full_analysis(test_text)


🏛️ [BERT 법률 문맥 분석 결과]
📊 예상 승소율: 0.0%
⚠️ 위험도 점수: 0.3/100


In [ ]:
# --- 샘플 테스트 실행 ---
print("\n--- 🔍 모델 성능 테스트 ---")
sample_text = "피고인은 피해자의 물건을 절취하였으며 이를 목격한 증인이 존재함에도 불구하고 범행을 부인하고 있다."
predict_full_analysis(sample_text)

sample_text_2 = "본 조항은 국민의 거주 이전의 자유를 과도하게 침해하여 헌법에 위배될 소지가 있습니다."
predict_full_analysis(sample_text_2)


--- 🔍 모델 성능 테스트 ---

🏛️ [BERT 법률 문맥 분석 결과]
📊 예상 승소율: 0.0%
⚠️ 위험도 점수: 0.5/100

🏛️ [BERT 법률 문맥 분석 결과]
📊 예상 승소율: 0.0%
⚠️ 위험도 점수: 0.4/100


In [ ]:
# 테스트 데이터로 실제 예측해보기
def evaluate_model_performance():
    from sklearn.metrics import classification_report, confusion_matrix
    import numpy as np
    
    model.eval()
    all_preds = []
    all_labels = []
    
    for i in range(min(100, len(test_dataset))):  # 100개 샘플만
        item = test_dataset[i]
        inputs = {k: v.unsqueeze(0).to(device) for k, v in item.items() 
                  if k in ['input_ids', 'attention_mask']}
        
        with torch.no_grad():
            outputs = model(**inputs)
            pred = outputs['logits'].argmax(-1).item()
            all_preds.append(pred)
            all_labels.append(item['labels'].item())
    
    # 상세 리포트
    class_names = train_df['case_type'].cat.categories.tolist()
    print("\n📊 상세 성능 리포트:")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    
    print("\n🎯 혼동 행렬 (Confusion Matrix):")
    print(confusion_matrix(all_labels, all_preds))

# 실행
evaluate_model_performance()

In [ ]:
# 학습 데이터 리포트
train_report = ProfileReport(train_df, title="train_deep_Report")
train_report.to_file("../report/train_deep_Report.html")

# 테스트 데이터 리포트
test_report = ProfileReport(test_df, title="Test_deep_Report")
test_report.to_file("../report/Test_deep_Report.html")

print("리포트 생성 완료! train_data_report.html 파일을 확인하세요.")